<a href="https://colab.research.google.com/github/akhilakaleru/Task3-ML/blob/main/TASK2_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
file_path = '/content/Dataset .csv'
data = pd.read_csv(file_path)

In [ ]:
irrelevant_columns = ["Restaurant ID", "Restaurant Name", "Address", "Locality", "Locality Verbose"]
data = data.drop(columns=irrelevant_columns)

In [ ]:
data["Cuisines"] = data["Cuisines"].fillna("Unknown")

In [ ]:
data["combined_features"] = (
    data["Cuisines"] + " " +
    data["Price range"].astype(str) + " " +
    data["Aggregate rating"].astype(str)
)

In [ ]:
vectorizer = CountVectorizer()
feature_matrix = vectorizer.fit_transform(data["combined_features"])

In [ ]:
cosine_sim = cosine_similarity(feature_matrix)

In [ ]:
def recommend_restaurants(preferred_cuisine, price_range, min_rating, top_n=5):
    filtered_data = data[
        (data["Cuisines"].str.contains(preferred_cuisine, case=False, na=False)) &
        (data["Price range"] == price_range) &
        (data["Aggregate rating"] >= min_rating)
    ]
    if filtered_data.empty:
        return "No restaurants match the given preferences."
    filtered_indices = filtered_data.index.tolist()
    similarity_scores = cosine_sim[filtered_indices].mean(axis=0)
    ranked_indices = similarity_scores.argsort()[::-1][:top_n]
    recommendations = data.iloc[ranked_indices]
    return recommendations[["City", "Cuisines", "Price range", "Aggregate rating"]]


In [ ]:
preferred_cuisine = "Japanese"
price_range = 3
min_rating = 4.0

In [ ]:
recommendations = recommend_restaurants(preferred_cuisine, price_range, min_rating, top_n=5)
print("Recommended Restaurants:")
print(recommendations)

Recommended Restaurants:
                City  Cuisines  Price range  Aggregate rating
1188         Gurgaon  Japanese            3               2.8
29         Bras�_lia  Japanese            4               3.7
429   Rest of Hawaii  Japanese            1               4.9
5417       New Delhi  Japanese            3               0.0
4335       New Delhi  Japanese            3               3.3
